# 28_02 고장 예측 결과 해석

In [ ]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import os
import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np


try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

Duplicate key in file WindowsPath('c:/Users/mzlap/Desktop/KNA-Data-analysis-1st/.venv/Lib/site-packages/matplotlib/mpl-data/matplotlibrc'), line 851 ('font.family : Malgun Gothic')
Duplicate key in file WindowsPath('c:/Users/mzlap/Desktop/KNA-Data-analysis-1st/.venv/Lib/site-packages/matplotlib/mpl-data/matplotlibrc'), line 852 ('axes.unicode_minus : False')


✅ 환경 설정 완료! 현재 적용된 폰트: ['Malgun Gothic']


In [2]:
# 28_1장 기본 머닝러신 코드 재현
df = pd.read_csv("28_cmapss_fd001_sample.csv")
feature_cols = ["sensor_2",	"sensor_3",	"sensor_4",	"sensor_7",	"sensor_11",	"sensor_15"]

X = df[feature_cols]
y = df["failure_soon"]

X.head()
# 분할은 2:8



,sensor_2,sensor_3,sensor_4,sensor_7,sensor_11,sensor_15
0,643.239,1598.756,1409.063,546.905,48.021,8.411
1,644.115,1589.055,1404.093,547.132,46.564,8.449
2,643.177,1593.635,1401.598,545.802,48.356,8.731
3,644.941,1593.509,1419.771,548.299,48.153,8.342
4,643.050,1614.132,1416.601,546.202,48.013,8.524


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42).fit(X_train , y_train)

model 

# y_pred = model.predict(X_test)

# y_pred[:15]

UnicodeDecodeError: 'cp949' codec can't decode byte 0xe2 in position 2950: illegal multibyte sequence

UnicodeDecodeError: 'cp949' codec can't decode byte 0xe2 in position 2950: illegal multibyte sequence

RandomForestClassifier(random_state=42)

# 01 F1-score와 분류 리포트
F1-score란 무엇인가 · 분류 리포트 읽는 순서


### F1 출력
고장 임박 클래스의 F1을 함수 한 줄로 계산해 출력


In [4]:
# 코드

from sklearn.metrics import f1_score

f1 = f1_score(y_test, y_pred)
print("고장임박 F1 :", round(f1, 4))



고장임박 F1 : 0.675


### 리포트 출력과 해석
클래스별 지표를 표로 출력하고 한 줄씩 해석 적기


In [5]:
# 코드

from sklearn.metrics import classification_report

# report = classification_report(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names = ["정상", "고장임박"])

print(report)

              precision    recall  f1-score   support

          정상       0.91      0.93      0.92       158
        고장임박       0.71      0.64      0.68        42

    accuracy                           0.87       200
   macro avg       0.81      0.79      0.80       200
weighted avg       0.87      0.87      0.87       200



# 02 임계값 조정
임계값을 조정한다는 것 · 임계값과 정밀도·재현율


### 확률 추출과 임계값 적용
고장(1) 확률을 꺼내 임계값별 예측 생성


In [6]:
# 코드

### 임계값별 지표 비교
각 임계값의 재현율·정밀도·F1 출력


In [7]:
# 코드

### 비교표 생성
임계값별 지표와 FP·FN을 한 표로


In [8]:
# 코드

# 03 이상탐지 평가와 결과 해석
이상탐지 결과 평가 · 정비 의사결정 해석


### IsolationForest 실행
MIMII 특징으로 이상탐지 학습·예측


In [9]:
# 코드

### 변환과 평가
-1을 1(이상)로 바꿔 라벨과 비교


In [10]:
# 코드

### MIMII 로드와 분할
MIMII 특징과 라벨로 학습 준비 — 지도학습 모드


In [11]:
# 코드

### 학습과 평가
RandomForest 학습 후 리포트 출력 — 동일 절차


In [12]:
# 코드

### 평가 함수 정의
모델과 평가 데이터를 받는 `evaluate()` 함수


In [13]:
# 코드

### 함수로 한 번에 평가
함수에 모델을 넣어 평가 실행 — 모델만 바꿔도 작동


In [14]:
# 코드

### 두 임계값 예측 생성
임계값 0.5와 0.3으로 예측을 만들고 재현율 비교


In [15]:
# 코드

### 리포트 초안 프롬프트
평가 수치를 정비 리포트 초안으로 정리하는 AI 프롬프트


In [16]:
# 코드